# KD Experiments Comparison - Detailed Analysis

This notebook compares all available KD experiments (`exp1_kkd_base` to `exp14_final_clustered`) using **all runs found on disk**.

For each experiment, the notebook:
- Aggregates the final metrics across all runs
- Reports mean and sample standard deviation (`std`)
- Exports detailed summaries to `kdexperiments_summary.txt` (matching FedAvg format exactly)

**KD Configuration:** Response-based Knowledge Distillation with:
- `distillation_alpha=0.5` (50% hard labels + 50% soft targets)
- `distillation_temperature=2.0`
- Teacher = global model, Student = local model (no gradient transfer)

In [12]:
from pathlib import Path
import pandas as pd
import json
import numpy as np
from IPython.display import Markdown, display

from reporting import (
    DISPLAY_METRICS,
    EXPERIMENT_ORDER,
    GLOBAL_SORT_ASCENDING,
    GLOBAL_SORT_COLUMNS,
    LOCAL_SORT_ASCENDING,
    LOCAL_SORT_COLUMNS,
    SUMMARY_TXT_PATH,
    aggregate_local_by_client,
    aggregate_local_rounds,
    aggregate_runs,
    format_mean_std,
    load_run_level_results,
)

cwd = Path.cwd()
if (cwd / 'config.py').exists() and (cwd / 'run_experiment.py').exists():
    kd_root = cwd
elif (cwd / 'kd-experiments').exists():
    kd_root = cwd / 'kd-experiments'
else:
    raise FileNotFoundError('Could not locate the kd-experiments folder from the current working directory.')

print(f'Using kd root: {kd_root}')

# Load results
results_df, local_by_client_df, local_round_df = load_run_level_results(kd_root)
aggregated_df = aggregate_runs(results_df)
aggregated_local_clients_df = aggregate_local_by_client(local_by_client_df)
aggregated_local_rounds_df = aggregate_local_rounds(local_round_df)

print(f"Loaded {len(results_df)} run records")
print(f"Aggregated into {len(aggregated_df)} experiment variants")
display(aggregated_df.head())

Using kd root: c:\Users\leono\Desktop\iscte\Tese\FL_MasterThesis\kd-experiments
Loaded 140 run records
Aggregated into 14 experiment variants


,comparison_id,n_runs,experiment_id,scenario,num_rounds,fraction_fit,min_fit_clients,min_available_clients,local_epochs,weighted_aggregation,...,fine_tune_miss_rate_mean,fine_tune_miss_rate_std,fine_tune_tn_mean,fine_tune_tn_std,fine_tune_fp_mean,fine_tune_fp_std,fine_tune_fn_mean,fine_tune_fn_std,fine_tune_tp_mean,fine_tune_tp_std
0,exp1_kkd_base,10,exp1_kkd_base,byclient,25.0,1.0,2.0,2.0,10.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,exp2_fraction_clients,10,exp2_fraction_clients,byclient,25.0,0.5,2.0,2.0,10.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,exp3_local_epochs,10,exp3_local_epochs,byclient,25.0,1.0,2.0,2.0,10.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,exp4_unweighted_aggregation,10,exp4_unweighted_aggregation,byclient,25.0,1.0,2.0,2.0,10.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,exp5_cross_dataset,10,exp5_cross_dataset,crossdataset,25.0,1.0,2.0,2.0,10.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Generate Detailed Summary (FedAvg Format)

In [13]:
# Define experiment metadata for detailed output
EXPERIMENT_METADATA = {
    'exp1_kkd_base': {
        'title': 'Experiment 1 - KD Base',
        'description': 'Baseline with Knowledge Distillation enabled (25 rounds, 5 epochs, weighted aggregation).'
    },
    'exp2_fraction_clients': {
        'title': 'Experiment 2 - Fraction Of Clients (KD)',
        'description': 'KD with partial client participation (50%) per round.'
    },
    'exp3_local_epochs': {
        'title': 'Experiment 3 - Local Epochs Comparison (KD)',
        'description': 'KD comparison across different numbers of local epochs (10, 25, 50, 75, 100).'
    },
    'exp4_unweighted_aggregation': {
        'title': 'Experiment 4 - Unweighted Aggregation (KD)',
        'description': 'KD variant with unweighted averaging across participating clients.'
    },
    'exp5_cross_dataset': {
        'title': 'Experiment 5 - Cross Dataset Federated (KD)',
        'description': 'KD with federated training on two datasets and evaluation on held-out dataset.'
    },
    'exp6_keep_best_local_model': {
        'title': 'Experiment 6 - Keep Best Local Model (KD)',
        'description': 'KD where each client keeps the better model between incoming global and previous best local.'
    },
    'exp7_clustered_aggregation': {
        'title': 'Experiment 7 - Clustered Aggregation (KD)',
        'description': 'KD with K-means clustering of similar clients before aggregation.'
    },
    'exp8_personalized_fedavg': {
        'title': 'Experiment 8 - Personalized FedAvg (KD)',
        'description': 'KD with personalized local head kept on each client (not aggregated).'
    },
    'exp9_final_local_finetuning': {
        'title': 'Experiment 9 - Final Local Fine-Tuning (KD)',
        'description': 'KD followed by short local fine-tuning stage on held-out test clients.'
    },
    'exp10_clustered_keep_best_local': {
        'title': 'Experiment 10 - Clustered Keep-Best Local (KD)',
        'description': 'KD combining clustering + local selection between incoming and previous best local model.'
    },
    'exp11_baseline_final': {
        'title': 'Experiment 11 - Baseline Final (KD)',
        'description': 'KD final baseline with 100 global rounds, 5 local epochs, weighted aggregation (matches FedAvg exp11).'
    },
    'exp12_final_unweighted': {
        'title': 'Experiment 12 - Final Unweighted (KD)',
        'description': 'KD with 100 global rounds, 5 local epochs, unweighted aggregation (matches FedAvg exp12 best result).'
    },
    'exp13_final_keep_best_local': {
        'title': 'Experiment 13 - Final Keep-Best Local (KD)',
        'description': 'KD with 100 global rounds, weighted aggregation, local model selection strategy.'
    },
    'exp14_final_clustered': {
        'title': 'Experiment 14 - Final Clustered (KD)',
        'description': 'KD with 100 global rounds, weighted aggregation, clustering strategy.'
    },
}

print("Generating detailed summary for all KD experiments...")
print("=" * 80)

Generating detailed summary for all KD experiments...


In [14]:
# Generate summary text file
summary_lines = ['KD Experiments Summary', '==========================', '']
summary_lines.append('Results are aggregated across all available runs for each experiment/variant.')
summary_lines.append('Uncertainty is reported as sample standard deviation across runs (mean +/- std).')
summary_lines.append('')
summary_lines.append('Knowledge Distillation Configuration:')
summary_lines.append('- Type: Response-based KD (output distribution matching)')
summary_lines.append('- Loss: alpha * BCE(hard_labels) + (1-alpha) * KL_divergence(soft_targets)')
summary_lines.append('- Parameters: distillation_alpha=0.5, distillation_temperature=2.0')
summary_lines.append('- Teacher: Global model (previous round)')
summary_lines.append('- Student: Local model being trained')
summary_lines.append('')

# Use comparison_id as the experiment identifier
if not aggregated_df.empty and 'comparison_id' in aggregated_df.columns:
    sorted_exp_ids = sorted([eid for eid in aggregated_df['comparison_id'].unique() if pd.notna(eid)])
    
    for exp_id in sorted_exp_ids:
        exp_data = aggregated_df[aggregated_df['comparison_id'] == exp_id]
        if len(exp_data) == 0:
            continue
        exp_df = exp_data.iloc[0]
        
        summary_lines.append(f"\n{exp_id}")
        summary_lines.append('-' * len(exp_id))
        
        # Get metadata
        metadata = EXPERIMENT_METADATA.get(exp_id, {})
        summary_lines.append(f"Title: {metadata.get('title', exp_id)}")
        summary_lines.append(f"Description: {metadata.get('description', 'N/A')}")
        
        # Runs
        n_runs = int(exp_df.get('n_runs', 0)) if 'n_runs' in exp_df.index else 0
        summary_lines.append(f"Runs aggregated: {n_runs}")
        
        # Configuration - extract from columns
        summary_lines.append(f"Global rounds: {int(exp_df.get('num_rounds', 25)) if 'num_rounds' in exp_df.index else 25}")
        summary_lines.append(f"Local epochs: {int(exp_df.get('local_epochs', 5)) if 'local_epochs' in exp_df.index else 5}")
        summary_lines.append(f"Weighted aggregation: {int(exp_df.get('weighted_aggregation', 1)) if 'weighted_aggregation' in exp_df.index else 1}")
        summary_lines.append(f"Local model selection: {int(exp_df.get('local_model_selection', 0)) if 'local_model_selection' in exp_df.index else 0}")
        summary_lines.append(f"Clustered aggregation: {int(exp_df.get('clustered_aggregation', 0)) if 'clustered_aggregation' in exp_df.index else 0}")
        summary_lines.append(f"Personalized head: {int(exp_df.get('personalized_head', 0)) if 'personalized_head' in exp_df.index else 0}")
        
        # Model config
        summary_lines.append(f"Model hidden layers: [256, 128, 64]")
        summary_lines.append(f"Model dropout: 0.3")
        summary_lines.append(f"Model learning rate: 0.0008")
        summary_lines.append(f"Model weight decay: 0.0001")
        summary_lines.append(f"Model batch size: 256.0")
        
        # Dev metrics
        summary_lines.append('')
        summary_lines.append('Final global dev metrics:')
        dev_metrics = ['accuracy', 'balanced_accuracy', 'specificity', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'far', 'miss_rate']
        for metric in dev_metrics:
            mean_col = f'dev_{metric}_mean'
            std_col = f'dev_{metric}_std'
            if mean_col in exp_df.index and std_col in exp_df.index:
                mean_val = exp_df[mean_col]
                std_val = exp_df[std_col]
                if pd.notna(mean_val) and pd.notna(std_val):
                    summary_lines.append(f"  {metric}: {float(mean_val):.4f} +/- {float(std_val):.4f}")
        
        # Test metrics
        summary_lines.append('')
        summary_lines.append('Final global test metrics:')
        for metric in dev_metrics:
            mean_col = f'test_{metric}_mean'
            std_col = f'test_{metric}_std'
            if mean_col in exp_df.index and std_col in exp_df.index:
                mean_val = exp_df[mean_col]
                std_val = exp_df[std_col]
                if pd.notna(mean_val) and pd.notna(std_val):
                    summary_lines.append(f"  {metric}: {float(mean_val):.4f} +/- {float(std_val):.4f}")
        
        # Fine-tuned metrics (if applicable)
        summary_lines.append('')
        summary_lines.append('Final fine-tuned test-client metrics:')
        fine_tune_metrics = ['accuracy', 'balanced_accuracy', 'specificity', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'far', 'miss_rate']
        has_fine_tune = False
        for metric in fine_tune_metrics:
            mean_col = f'fine_tune_{metric}_mean'
            if mean_col in exp_df.index and pd.notna(exp_df[mean_col]):
                has_fine_tune = True
                break
        
        if has_fine_tune:
            for metric in fine_tune_metrics:
                mean_col = f'fine_tune_{metric}_mean'
                std_col = f'fine_tune_{metric}_std'
                if mean_col in exp_df.index and std_col in exp_df.index:
                    mean_val = exp_df[mean_col]
                    std_val = exp_df[std_col]
                    if pd.notna(mean_val) and pd.notna(std_val):
                        summary_lines.append(f"  {metric}: {float(mean_val):.4f} +/- {float(std_val):.4f}")
        else:
            summary_lines.append('  n/a')
else:
    summary_lines.append('No results available yet.')

# Write to file
summary_path = kd_root / 'kdexperiments_summary.txt'
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')

print(f"\n✓ Summary written to: {summary_path}")
print(f"\nPreview (first 80 lines):")
display(Markdown('```text\n' + '\n'.join(summary_lines[:80]) + '\n```'))


✓ Summary written to: c:\Users\leono\Desktop\iscte\Tese\FL_MasterThesis\kd-experiments\kdexperiments_summary.txt

Preview (first 80 lines):


```text
KD Experiments Summary
==========================

Results are aggregated across all available runs for each experiment/variant.
Uncertainty is reported as sample standard deviation across runs (mean +/- std).

Knowledge Distillation Configuration:
- Type: Response-based KD (output distribution matching)
- Loss: alpha * BCE(hard_labels) + (1-alpha) * KL_divergence(soft_targets)
- Parameters: distillation_alpha=0.5, distillation_temperature=2.0
- Teacher: Global model (previous round)
- Student: Local model being trained


exp10_clustered_keep_best_local
-------------------------------
Title: Experiment 10 - Clustered Keep-Best Local (KD)
Description: KD combining clustering + local selection between incoming and previous best local model.
Runs aggregated: 10
Global rounds: 25
Local epochs: 10
Weighted aggregation: 1
Local model selection: 1
Clustered aggregation: 1
Personalized head: 0
Model hidden layers: [256, 128, 64]
Model dropout: 0.3
Model learning rate: 0.0008
Model weight decay: 0.0001
Model batch size: 256.0

Final global dev metrics:
  accuracy: 0.8975 +/- 0.0034
  balanced_accuracy: 0.8897 +/- 0.0029
  specificity: 0.9094 +/- 0.0063
  precision: 0.8053 +/- 0.0101
  recall: 0.8700 +/- 0.0073
  f1: 0.8363 +/- 0.0044
  roc_auc: 0.9546 +/- 0.0016
  pr_auc: 0.9091 +/- 0.0039
  far: 0.0906 +/- 0.0063
  miss_rate: 0.1300 +/- 0.0073

Final global test metrics:
  accuracy: 0.9206 +/- 0.0029
  balanced_accuracy: 0.9209 +/- 0.0027
  specificity: 0.9202 +/- 0.0034
  precision: 0.8392 +/- 0.0059
  recall: 0.9217 +/- 0.0032
  f1: 0.8785 +/- 0.0041
  roc_auc: 0.9742 +/- 0.0009
  pr_auc: 0.9487 +/- 0.0031
  far: 0.0798 +/- 0.0034
  miss_rate: 0.0783 +/- 0.0032

Final fine-tuned test-client metrics:
  n/a

exp11_baseline_final
--------------------
Title: Experiment 11 - Baseline Final (KD)
Description: KD final baseline with 100 global rounds, 5 local epochs, weighted aggregation (matches FedAvg exp11).
Runs aggregated: 10
Global rounds: 25
Local epochs: 100
Weighted aggregation: 1
Local model selection: 0
Clustered aggregation: 0
Personalized head: 0
Model hidden layers: [256, 128, 64]
Model dropout: 0.3
Model learning rate: 0.0008
Model weight decay: 0.0001
Model batch size: 256.0

Final global dev metrics:
  accuracy: 0.8942 +/- 0.0038
  balanced_accuracy: 0.8929 +/- 0.0029
  specificity: 0.8962 +/- 0.0072
  precision: 0.7869 +/- 0.0107
  recall: 0.8895 +/- 0.0078
  f1: 0.8350 +/- 0.0046
```

In [15]:
# Show experiment availability
print("Experiment Availability:")
print(f"Total runs loaded: {len(results_df)}")
print(f"\nRuns per experiment:")
if 'experiment_id' in results_df.columns:
    exp_counts = results_df['experiment_id'].value_counts().sort_index()
    print(exp_counts)
else:
    print("  (experiment_id column not found)")

print(f"\nAggregated experiments: {len(aggregated_df)}")
if not aggregated_df.empty:
    display_cols = [col for col in ['comparison_id', 'experiment_id', 'n_runs'] if col in aggregated_df.columns]
    if display_cols:
        display(aggregated_df[display_cols].head(20))

Experiment Availability:
Total runs loaded: 140

Runs per experiment:
experiment_id
exp10_clustered_keep_best_local    10
exp11_baseline_final               10
exp12_final_unweighted             10
exp13_final_keep_best_local        10
exp14_final_clustered              10
exp1_kkd_base                      10
exp2_fraction_clients              10
exp3_local_epochs                  10
exp4_unweighted_aggregation        10
exp5_cross_dataset                 10
exp6_keep_best_local_model         10
exp7_clustered_aggregation         10
exp8_personalized_fedavg           10
exp9_final_local_finetuning        10
Name: count, dtype: int64

Aggregated experiments: 14


,comparison_id,experiment_id,n_runs
0,exp1_kkd_base,exp1_kkd_base,10
1,exp2_fraction_clients,exp2_fraction_clients,10
2,exp3_local_epochs,exp3_local_epochs,10
3,exp4_unweighted_aggregation,exp4_unweighted_aggregation,10
4,exp5_cross_dataset,exp5_cross_dataset,10
5,exp6_keep_best_local_model,exp6_keep_best_local_model,10
6,exp7_clustered_aggregation,exp7_clustered_aggregation,10
7,exp8_personalized_fedavg,exp8_personalized_fedavg,10
8,exp9_final_local_finetuning,exp9_final_local_finetuning,10
9,exp10_clustered_keep_best_local,exp10_clustered_keep_best_local,10


## Global Test Performance Ranking

In [16]:
global_rows = []
for _, row in aggregated_df.iterrows():
    global_rows.append({
        'experiment_id': row['experiment_id'],
        'comparison_id': row['comparison_id'],
        'title': row.get('title', ''),
        'n_runs': int(row.get('n_runs', 0)),
        'scenario': row.get('scenario'),
        'num_rounds': row.get('num_rounds'),
        'local_epochs': row.get('local_epochs'),
        'test_accuracy': format_mean_std(row.get('test_accuracy_mean'), row.get('test_accuracy_std')),
        'test_balanced_accuracy': format_mean_std(row.get('test_balanced_accuracy_mean'), row.get('test_balanced_accuracy_std')),
        'test_f1': format_mean_std(row.get('test_f1_mean'), row.get('test_f1_std')),
        'test_pr_auc': format_mean_std(row.get('test_pr_auc_mean'), row.get('test_pr_auc_std')),
        'test_roc_auc': format_mean_std(row.get('test_roc_auc_mean'), row.get('test_roc_auc_std')),
        'test_far': format_mean_std(row.get('test_far_mean'), row.get('test_far_std')),
        'test_miss_rate': format_mean_std(row.get('test_miss_rate_mean'), row.get('test_miss_rate_std')),
    })

global_ranking_df = pd.DataFrame(global_rows)

# Sort by PR-AUC (primary metric for imbalanced classification)
if 'test_pr_auc_mean' in aggregated_df.columns:
    sorted_idx = aggregated_df['test_pr_auc_mean'].fillna(-1).argsort()[::-1]
    global_ranking_df = global_ranking_df.iloc[sorted_idx.values].reset_index(drop=True)

display(global_ranking_df)

,experiment_id,comparison_id,title,n_runs,scenario,num_rounds,local_epochs,test_accuracy,test_balanced_accuracy,test_f1,test_pr_auc,test_roc_auc,test_far,test_miss_rate
0,exp12_final_unweighted,exp12_final_unweighted,,10,byclient,25.0,100.0,0.9111 ± 0.0057,0.9202 ± 0.0035,0.8687 ± 0.0069,0.9586 ± 0.0025,0.9779 ± 0.0009,0.1039 ± 0.0096,0.0556 ± 0.0039
1,exp13_final_keep_best_local,exp13_final_keep_best_local,,10,byclient,25.0,100.0,0.9208 ± 0.0039,0.9247 ± 0.0029,0.8803 ± 0.0052,0.9550 ± 0.0028,0.9768 ± 0.0009,0.0857 ± 0.0060,0.0649 ± 0.0035
2,exp11_baseline_final,exp11_baseline_final,,10,byclient,25.0,100.0,0.9210 ± 0.0039,0.9249 ± 0.0029,0.8806 ± 0.0051,0.9547 ± 0.0020,0.9767 ± 0.0007,0.0855 ± 0.0061,0.0647 ± 0.0040
3,exp14_final_clustered,exp14_final_clustered,,10,byclient,25.0,100.0,0.9208 ± 0.0040,0.9247 ± 0.0026,0.8802 ± 0.0052,0.9544 ± 0.0020,0.9766 ± 0.0007,0.0857 ± 0.0065,0.0648 ± 0.0023
4,exp4_unweighted_aggregation,exp4_unweighted_aggregation,,10,byclient,25.0,10.0,0.9141 ± 0.0035,0.9182 ± 0.0024,0.8707 ± 0.0044,0.9518 ± 0.0034,0.9749 ± 0.0010,0.0927 ± 0.0065,0.0708 ± 0.0056
5,exp2_fraction_clients,exp2_fraction_clients,,10,byclient,25.0,10.0,0.9204 ± 0.0017,0.9208 ± 0.0018,0.8781 ± 0.0022,0.9494 ± 0.0018,0.9745 ± 0.0007,0.0804 ± 0.0041,0.0780 ± 0.0061
6,exp1_kkd_base,exp1_kkd_base,,10,byclient,25.0,10.0,0.9209 ± 0.0027,0.9212 ± 0.0022,0.8789 ± 0.0037,0.9491 ± 0.0010,0.9745 ± 0.0006,0.0796 ± 0.0042,0.0780 ± 0.0036
7,exp9_final_local_finetuning,exp9_final_local_finetuning,,10,byclient,25.0,10.0,0.9208 ± 0.0025,0.9210 ± 0.0020,0.8788 ± 0.0034,0.9490 ± 0.0028,0.9744 ± 0.0009,0.0795 ± 0.0043,0.0784 ± 0.0039
8,exp3_local_epochs,exp3_local_epochs,,10,byclient,25.0,10.0,0.9204 ± 0.0024,0.9206 ± 0.0018,0.8781 ± 0.0032,0.9489 ± 0.0028,0.9743 ± 0.0009,0.0800 ± 0.0039,0.0788 ± 0.0030
9,exp6_keep_best_local_model,exp6_keep_best_local_model,,10,byclient,25.0,10.0,0.9209 ± 0.0038,0.9204 ± 0.0030,0.8785 ± 0.0051,0.9487 ± 0.0034,0.9743 ± 0.0010,0.0784 ± 0.0057,0.0808 ± 0.0040


## Validation Set Performance During Training

In [17]:
local_rows = []
for _, row in aggregated_df.iterrows():
    if pd.isna(row.get('local_pr_auc_mean')):
        continue
    local_rows.append({
        'experiment_id': row['experiment_id'],
        'comparison_id': row['comparison_id'],
        'title': row.get('title', ''),
        'n_runs': int(row.get('n_runs', 0)),
        'local_accuracy': format_mean_std(row.get('local_accuracy_mean'), row.get('local_accuracy_std')),
        'local_balanced_accuracy': format_mean_std(row.get('local_balanced_accuracy_mean'), row.get('local_balanced_accuracy_std')),
        'local_f1': format_mean_std(row.get('local_f1_mean'), row.get('local_f1_std')),
        'local_pr_auc': format_mean_std(row.get('local_pr_auc_mean'), row.get('local_pr_auc_std')),
        'local_roc_auc': format_mean_std(row.get('local_roc_auc_mean'), row.get('local_roc_auc_std')),
        'local_far': format_mean_std(row.get('local_far_mean'), row.get('local_far_std')),
        'local_miss_rate': format_mean_std(row.get('local_miss_rate_mean'), row.get('local_miss_rate_std')),
    })

local_ranking_df = pd.DataFrame(local_rows)

# Sort by PR-AUC
if 'local_pr_auc_mean' in aggregated_df.columns:
    sorted_idx = aggregated_df.dropna(subset=['local_pr_auc_mean'])['local_pr_auc_mean'].argsort()[::-1]
    if len(sorted_idx) > 0:
        local_ranking_df = local_ranking_df.iloc[sorted_idx.values].reset_index(drop=True)

display(local_ranking_df if not local_ranking_df.empty else pd.DataFrame(columns=['experiment_id']))

,experiment_id


## Experiment 3: Local Epochs Convergence Study

In [18]:
exp3_df = aggregated_df[aggregated_df['experiment_id'] == 'exp3_local_epochs'].copy() if not aggregated_df.empty else pd.DataFrame()
if not exp3_df.empty:
    exp3_table = exp3_df[[
        'comparison_id', 'local_epochs', 'n_runs',
        'test_pr_auc_mean', 'test_pr_auc_std',
        'test_f1_mean', 'test_f1_std',
        'test_balanced_accuracy_mean', 'test_balanced_accuracy_std',
        'test_far_mean', 'test_far_std',
        'test_miss_rate_mean', 'test_miss_rate_std',
    ]].copy()
    exp3_table['test_pr_auc'] = exp3_table.apply(lambda r: format_mean_std(r['test_pr_auc_mean'], r['test_pr_auc_std']), axis=1)
    exp3_table['test_f1'] = exp3_table.apply(lambda r: format_mean_std(r['test_f1_mean'], r['test_f1_std']), axis=1)
    exp3_table['test_balanced_accuracy'] = exp3_table.apply(lambda r: format_mean_std(r['test_balanced_accuracy_mean'], r['test_balanced_accuracy_std']), axis=1)
    exp3_table['test_far'] = exp3_table.apply(lambda r: format_mean_std(r['test_far_mean'], r['test_far_std']), axis=1)
    exp3_table['test_miss_rate'] = exp3_table.apply(lambda r: format_mean_std(r['test_miss_rate_mean'], r['test_miss_rate_std']), axis=1)
    exp3_table = exp3_table[[
        'comparison_id', 'local_epochs', 'n_runs',
        'test_pr_auc', 'test_f1', 'test_balanced_accuracy', 'test_far', 'test_miss_rate'
    ]].sort_values('local_epochs').reset_index(drop=True)
    display(exp3_table)
else:
    print("No exp3_local_epochs data found")

,comparison_id,local_epochs,n_runs,test_pr_auc,test_f1,test_balanced_accuracy,test_far,test_miss_rate
0,exp3_local_epochs,10.0,10,0.9489 ± 0.0028,0.8781 ± 0.0032,0.9206 ± 0.0018,0.0800 ± 0.0039,0.0788 ± 0.0030


## Metrics By Round (Federated Training Dynamics)

In [19]:
round_columns = [
    'comparison_id', 'experiment_id', 'round',
    'val_pr_auc_mean', 'val_pr_auc_std',
    'val_f1_mean', 'val_f1_std',
    'val_balanced_accuracy_mean', 'val_balanced_accuracy_std',
    'val_far_mean', 'val_far_std',
    'val_miss_rate_mean', 'val_miss_rate_std',
]
round_columns = [c for c in round_columns if c in aggregated_local_rounds_df.columns]
if not aggregated_local_rounds_df.empty:
    display(aggregated_local_rounds_df[round_columns].head(50))
    print(f"... (showing first 50 of {len(aggregated_local_rounds_df)} round records)")
else:
    print("No per-round metrics available")

,comparison_id,experiment_id,round,val_pr_auc_mean,val_pr_auc_std,val_f1_mean,val_f1_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_far_mean,val_far_std,val_miss_rate_mean,val_miss_rate_std
0,exp11_baseline_final,exp11_baseline_final,1,0.920686,0.007935,0.774321,0.008770,0.933411,0.003387,0.069720,0.012862,0.063458,0.014265
1,exp11_baseline_final,exp11_baseline_final,2,0.935118,0.005685,0.789700,0.006749,0.944143,0.002567,0.052567,0.005176,0.059147,0.007931
2,exp11_baseline_final,exp11_baseline_final,3,0.934911,0.005208,0.792272,0.006573,0.945637,0.002267,0.046809,0.003777,0.061917,0.005517
3,exp11_baseline_final,exp11_baseline_final,4,0.934532,0.004774,0.791602,0.006459,0.945474,0.002261,0.046699,0.004346,0.062256,0.005776
4,exp11_baseline_final,exp11_baseline_final,5,0.933585,0.004561,0.790866,0.006544,0.944631,0.002100,0.048249,0.004664,0.061595,0.005684
5,exp11_baseline_final,exp11_baseline_final,6,0.932359,0.005139,0.789664,0.006417,0.942839,0.002098,0.050310,0.005087,0.062063,0.005715
6,exp11_baseline_final,exp11_baseline_final,7,0.932286,0.004425,0.789153,0.006734,0.941682,0.002475,0.051613,0.005054,0.062256,0.005437
7,exp11_baseline_final,exp11_baseline_final,8,0.931858,0.004585,0.788683,0.007052,0.940735,0.002580,0.052829,0.006013,0.062198,0.006307
8,exp11_baseline_final,exp11_baseline_final,9,0.931174,0.004457,0.788302,0.007125,0.940033,0.002446,0.053670,0.005490,0.062229,0.005981
9,exp11_baseline_final,exp11_baseline_final,10,0.931082,0.004119,0.788192,0.006872,0.939800,0.002368,0.055234,0.005388,0.060744,0.005719


... (showing first 50 of 225 round records)


## Per-Client Metrics

In [20]:
client_columns = [
    'comparison_id', 'experiment_id', 'dataset', 'client', 'cluster_id',
    'accuracy_mean', 'accuracy_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    'f1_mean', 'f1_std',
    'pr_auc_mean', 'pr_auc_std',
]
client_columns = [c for c in client_columns if c in aggregated_local_clients_df.columns]
if not aggregated_local_clients_df.empty:
    display(aggregated_local_clients_df[client_columns].head(50))
    print(f"... (showing first 50 of {len(aggregated_local_clients_df)} client records)")
else:
    print("No per-client metrics available")

No per-client metrics available


## Summary: Best Performing KD Configurations

### Top-3 Experiments by Test PR-AUC (Primary Metric):

The ranking above shows the best KD variants. Compare against FedAvg results:
- **FedAvg exp12 (unweighted)**: PR-AUC = 0.9676 ± 0.0024, F1 = 0.9171 ± 0.0041
- **FedAvg exp11 (baseline)**: PR-AUC = 0.9658 ± 0.0020, F1 = 0.9166 ± 0.0032

### Expected KD Impact:
1. **Convergence**: KD should provide smoother validation curves via teacher regularization
2. **Performance**: May match or slightly exceed FedAvg due to knowledge regularization
3. **Variance**: With 10 runs per experiment, look for consistent performance across runs
4. **Local Epochs (exp3)**: Identify optimal local training intensity with KD
5. **Heterogeneity**: exp7/exp10/exp14 (clustering) should benefit more from KD regularization

Detailed results exported to: `kdexperiments_summary.txt` (matching FedAvg format)